In [1]:
import os
import textwrap

from langchain_core.globals import get_llm_cache, set_llm_cache
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_anthropic import ChatAnthropic
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI

import llm_cache_stats_wrapper
import simple_llm_cache

# In order to make it easy to run work projects and personal AI experiments, override these key values with the value of *_PERSONAL, if set.
if "OPENAI_API_KEY_PERSONAL" in os.environ:
    print("Using key from OPENAI_API_KEY_PERSONAL environment variable")
    os.environ["OPENAI_API_KEY"] = os.environ["OPENAI_API_KEY_PERSONAL"]
if "ANTHROPIC_API_KEY_PERSONAL" in os.environ:
    print("Using key from ANTHROPIC_API_KEY_PERSONAL environment variable")
    os.environ["ANTHROPIC_API_KEY"] = os.environ["ANTHROPIC_API_KEY_PERSONAL"]
if "GEMINI_API_KEY_PERSONAL" in os.environ:
    print("Using key from GEMINI_API_KEY_PERSONAL environment variable")
    os.environ["GEMINI_API_KEY"] = os.environ["GEMINI_API_KEY_PERSONAL"]

verbose = False
temperature = 0.5

set_llm_cache(llm_cache_stats_wrapper.LlmCacheStatsWrapper(simple_llm_cache.SimpleLlmCache("llm-cache.json")))

def dump_cache_stats_since_last_call():
    print(get_llm_cache().get_cache_stats_summary())
    get_llm_cache().clear_cache_stats()

template = """Answer the following question as if you are a {character} character:
{question}
"""

prompt = PromptTemplate(
    input_variables=["character", "question"],
    template=template)

for model_name in [
    # "gpt-3.5-turbo",
    # "gpt-4",
    # "gpt-4o-mini",
    "gpt-4o",
    # "claude-3-haiku-20240307",
    "gemini-3.5-flash-lite",
]:
    if model_name.startswith("gpt-"):
        llm = ChatOpenAI(
            temperature=temperature,
            model_name = model_name)
    elif model_name.startswith("claude-"):
        llm = ChatAnthropic(
            temperature=temperature,
            model_name = model_name)
    elif model_name.startswith("gemini-"):
        # Reads the key from the GEMINI_API_KEY environment variable.
        llm = ChatGoogleGenerativeAI(temperature=temperature, model=model_name)

    # LLMChain was removed; compose the prompt, model, and output parser with LCEL instead.
    chain = (prompt | llm | StrOutputParser()).with_config(verbose=verbose)

    for trial in range(2):
        print(f"\n*** {model_name} trial {trial} ***")
        get_llm_cache().inner_cache.set_trial(trial)
        output = chain.invoke({
            "character": "pirate",
            "question": "What is a neural network?"})
        print(textwrap.fill(output, width=80))

print()
dump_cache_stats_since_last_call()

Using key from OPENAI_API_KEY_PERSONAL environment variable
Using key from ANTHROPIC_API_KEY_PERSONAL environment variable
Using key from GEMINI_API_KEY_PERSONAL environment variable

*** gpt-4o trial 0 ***


/workspaces/MinSampleTemplates/LangChain/simple_llm_cache.py:116: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  generations.append(loads(generation))
/workspaces/MinSampleTemplates/LangChain/simple_llm_cache.py:116: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  generations.append(loads(generation))


Arrr, ye be askin' 'bout a neural network, eh? Well, let me spin ye a yarn,
matey! A neural network be like a crew o' smart sailors workin' together on a
ship, each with their own special job. These sailors be nodes, and they be
connected by ropes o' knowledge called connections.   The ship's mission be to
learn from the vast ocean o' data, adjustin' their sails and ropes to make sense
o' patterns and hidden treasures within. They take in information from one end,
pass it through layers o' thinkin', and spit out answers or predictions at the
other end.   Just like a pirate crew, they learn from their mistakes, adjustin'
their course until they get it right. So, a neural network be a mighty clever
contraption, helpin' us navigate the treacherous waters o' data and findin' the
booty o' knowledge hidden within! Arrr!

*** gpt-4o trial 1 ***
Arrr, ye be askin' about a neural network, eh? Well, let me spin ye a yarn. A
neural network be like a crew of scallywags aboard a ship, all workin' t

In [2]:
import asyncio


async def predict_async(trial):
    print(f"\n*** Calling ainvoke trial {trial}")
    get_llm_cache().inner_cache.set_trial(trial)
    output = await chain.ainvoke({"character": "poet", "question": "What is a neural network?"})
    print(f"\n*** Returned from ainvoke trial {trial}")
    print(output)
    print()


# Use the async method to demonstrate that the cache also works for async calls
coroutines = []
for trial in range(10):
    coroutine = predict_async(trial)
    coroutines.append(coroutine)

await asyncio.gather(*coroutines)


dump_cache_stats_since_last_call()


*** Calling ainvoke trial 0

*** Calling ainvoke trial 1

*** Calling ainvoke trial 2

*** Calling ainvoke trial 3

*** Calling ainvoke trial 4

*** Calling ainvoke trial 5

*** Calling ainvoke trial 6

*** Calling ainvoke trial 7

*** Calling ainvoke trial 8

*** Calling ainvoke trial 9

*** Returned from ainvoke trial 0
A loom of light, unseen, profound,
Where whispered echoes learn the sound
Of waking thought. A lattice spun
Of silver wire and autumn sun.

It knows no hand, it feels no pain,
Yet drinks the rivers of our brain.
A thousand nodes, like stars at night,
Are kindled by a spark of white.

They take the shadow, take the spark,
And trace their pathways through the dark—
A million guesses, falling low,
Until the sudden flowers grow.

Not bone, not breath, but glass and wire,
Yet breathing with a stolen fire;
We weave the ghost, and watch it rise,
To see ourselves through metal eyes.


*** Returned from ainvoke trial 1
Ah, you ask of the ghost within the wire,  
Of silicon ve